# Stage 9 - Entropy and Microstructure Features (AFML Ch. 18-19)

Analyses the entropy and microstructure features on the full daily series.
Features computed for modelling (195 event timestamps) are extended here
to the complete daily index for interpretability and report figures.

**Implements:** Shannon entropy, Lempel-Ziv complexity (Ch. 18);  
Corwin-Schultz spread, Amihud illiquidity, Bekker-Parkinson vol (Ch. 19)

**Outputs:** P23 (rolling entropy), P24 (Corwin-Schultz spread), correlation tables

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from entropy import rolling_entropy, rolling_lz, shannon_entropy, lempel_ziv_complexity
from microstructure import (corwin_schultz_spread, bekker_parkinson_vol,
                             amihud_illiquidity, roll_spread)

FIGURES = '../reports/figures'
plt.rcParams.update({'font.size': 11, 'grid.alpha': 0.3})
print('Imports OK')

Imports OK


## 1. Load full daily data and compute log returns

In [2]:
clean = pd.read_parquet('../data/processed/nvda_clean.parquet')
log_ret = np.log(clean['Adj Close'] / clean['Adj Close'].shift(1))
print(f'Daily data: {len(clean)} rows  {clean.index[0].date()} to {clean.index[-1].date()}')
print(f'Log return stats: mean={log_ret.mean():.5f}  std={log_ret.std():.4f}')

Daily data: 5113 rows  2005-01-04 to 2025-04-30
Log return stats: mean=0.00126  std=0.0309


## 2. Entropy features on full daily series

Rolling window = 50 trading days (≈2.5 months). AFML Ch. 18.

In [3]:
WINDOW = 50
ent_df = pd.DataFrame(index=clean.index)
ent_df['shannon_entropy'] = rolling_entropy(log_ret, window=WINDOW, n_bins=10)
ent_df['lempel_ziv']     = rolling_lz(log_ret, window=WINDOW)
print('Entropy features computed:')
print(ent_df.describe().round(4))

Entropy features computed:
       shannon_entropy  lempel_ziv
count        5063.0000   5064.0000
mean            1.8870      1.2492
std             0.2207      0.1040
min             0.9740      0.7901
25%             1.7884      1.2416
50%             1.9328      1.2416
75%             2.0329      1.3545
max             2.2831      1.5803


## 3. Microstructure features on full daily series

Corwin-Schultz spread, Bekker-Parkinson vol, Amihud illiquidity, Roll spread.  
All are daily-OHLCV approximations. AFML Ch. 19.

In [4]:
micro_df = pd.DataFrame(index=clean.index)
micro_df['cs_spread']    = corwin_schultz_spread(clean['High'], clean['Low'])
micro_df['bp_vol']       = bekker_parkinson_vol(clean['High'], clean['Low'])
micro_df['amihud']       = amihud_illiquidity(clean['Adj Close'], clean['Volume'], window=20)
micro_df['roll_spread']  = roll_spread(clean['Adj Close'], window=20)
print('Microstructure features:')
print(micro_df.describe().round(6))
print(f'\nCS spread zeros (clipped): {(micro_df.cs_spread==0).sum()} / {micro_df.cs_spread.notna().sum()}')

Microstructure features:
         cs_spread       bp_vol  amihud  roll_spread
count  5112.000000  5113.000000  5093.0  5092.000000
mean      0.007944     0.000625     0.0     0.224018
std       0.011352     0.000977     0.0     0.853209
min       0.000000     0.000016     0.0     0.000000
25%       0.000000     0.000175     0.0     0.000000
50%       0.001836     0.000348     0.0     0.005054
75%       0.013181     0.000692     0.0     0.037174
max       0.105515     0.029559     0.0     8.245012

CS spread zeros (clipped): 2411 / 5112


## 4. P23 - Rolling entropy time series

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), dpi=120, sharex=True)

ax = axes[0]
ax.plot(ent_df.index, ent_df['shannon_entropy'], color='purple', lw=0.8,
        label=f'Shannon entropy (nats, {WINDOW}-day window)')
ax.set_ylabel('Shannon entropy')
ax.set_title('Rolling Entropy Features  (AFML Ch. 18)', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True)

ax = axes[1]
ax.plot(ent_df.index, ent_df['lempel_ziv'], color='teal', lw=0.8,
        label=f'LZ-76 complexity (normalised, {WINDOW}-day window)')
ax.axhline(1.0, color='grey', ls='--', lw=0.8, label='Random benchmark = 1.0')
ax.set_ylabel('LZ complexity (normalised)')
ax.set_xlabel('Date')
ax.legend(fontsize=10); ax.grid(True)
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P23_rolling_entropy.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved P23')

Saved P23


## 5. P24 - Microstructure features time series

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), dpi=120, sharex=True)
axes = axes.flatten()

series_info = [
    ('cs_spread',   'Corwin-Schultz bid-ask spread',  'steelblue'),
    ('bp_vol',      'Bekker-Parkinson variance',       'darkorange'),
    ('amihud',      'Amihud illiquidity (20d mean)',   'green'),
    ('roll_spread', 'Roll spread estimate (20d)',      'red'),
]
for ax, (col, title, color) in zip(axes, series_info):
    valid = micro_df[col].dropna()
    ax.plot(valid.index, valid.values, color=color, lw=0.7)
    ax.set_title(title, fontsize=10)
    ax.grid(True)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Microstructure Features  (AFML Ch. 19 - daily OHLCV approximations)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES}/P24_microstructure_features.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved P24')

Saved P24


## 6. Correlation with forward 10-day return and labels

In [7]:
# Forward 10-day return (for correlation analysis, not used in model)
fwd_10d = np.log(clean['Adj Close'].shift(-10) / clean['Adj Close'])

# Load modelling dataset for labels at event timestamps
mod = pd.read_parquet('../data/processed/nvda_modelling_dataset.parquet')

# Align features to event timestamps
feat_at_events = pd.concat([
    ent_df, micro_df
], axis=1).loc[mod.index]

feat_at_events['fwd_10d'] = fwd_10d.reindex(mod.index)
feat_at_events['label']   = mod['label']

feat_cols = ['shannon_entropy', 'lempel_ziv', 'cs_spread', 'bp_vol', 'amihud', 'roll_spread']
corr_df = feat_at_events[feat_cols + ['fwd_10d', 'label']].corr()[['fwd_10d', 'label']].loc[feat_cols]
corr_df.columns = ['Corr with fwd_10d_return', 'Corr with label']
corr_df = corr_df.round(4)
print('Correlation of entropy/microstructure features with outcomes:')
print(corr_df.to_string())

Correlation of entropy/microstructure features with outcomes:
                 Corr with fwd_10d_return  Corr with label
shannon_entropy                   -0.1626          -0.1082
lempel_ziv                        -0.0016          -0.0867
cs_spread                          0.0206          -0.0262
bp_vol                             0.1513           0.0347
amihud                             0.0833           0.1948
roll_spread                        0.1160          -0.1204


## 7. Pairwise entropy-microstructure correlation heatmap

In [8]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
corr_mat = feat_at_events[feat_cols].corr()
im = ax.imshow(corr_mat.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(feat_cols))); ax.set_yticks(range(len(feat_cols)))
short_names = ['Shannon H', 'LZ', 'CS spread', 'BP vol', 'Amihud', 'Roll']
ax.set_xticklabels(short_names, rotation=45, ha='right')
ax.set_yticklabels(short_names)
plt.colorbar(im, ax=ax, label='Pearson r')
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        ax.text(j, i, f'{corr_mat.values[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Entropy & Microstructure Feature Correlations', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES}/P_entropy_micro_corr.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved entropy-microstructure correlation heatmap')

Saved entropy-microstructure correlation heatmap


## 8. Summary statistics table

In [9]:
summary = pd.DataFrame({
    'Shannon entropy': ent_df['shannon_entropy'].describe(),
    'LZ complexity':   ent_df['lempel_ziv'].describe(),
    'CS spread':       micro_df['cs_spread'].describe(),
    'Amihud illiq.':   micro_df['amihud'].describe(),
}).round(6)
print('Feature summary statistics (full daily series):')
print(summary.to_string())

# Correlation summary
print('\nCorrelation with labels and forward returns:')
print(corr_df.to_string())
print('\nNote: CS spread has many zero values due to floor-at-zero correction.')
print('This is a known limitation of Corwin-Schultz with daily OHLCV data.')

Feature summary statistics (full daily series):
       Shannon entropy  LZ complexity    CS spread  Amihud illiq.
count      5063.000000    5064.000000  5112.000000         5093.0
mean          1.887023       1.249205     0.007944            0.0
std           0.220661       0.103980     0.011352            0.0
min           0.974032       0.790140     0.000000            0.0
25%           1.788408       1.241648     0.000000            0.0
50%           1.932836       1.241648     0.001836            0.0
75%           2.032862       1.354525     0.013181            0.0
max           2.283128       1.580280     0.105515            0.0

Correlation with labels and forward returns:
                 Corr with fwd_10d_return  Corr with label
shannon_entropy                   -0.1626          -0.1082
lempel_ziv                        -0.0016          -0.0867
cs_spread                          0.0206          -0.0262
bp_vol                             0.1513           0.0347
amihud           